# Agent Transaction Data Cleaning and Quality Analysis

**Project:** PLP Data Analysis – Assignment 3

**Objective:** Audit, clean and validate agent transaction records, investigate data quality issues, and produce reliable county-level transaction summaries.

**Data Sources:**
- agents.csv
- agent_transactions.csv

**Tools:** Python, Pandas and NumPy

In [28]:

%%writefile cleaning_pipeline.py

import pandas as pd


def clean_agent_data(txns, agents):
    """
    Clean agent transaction data and return a cleaning log.

    Handles:
    - Duplicate transaction IDs
    - Inconsistent transaction types
    - Missing and invalid amounts
    - Date format inconsistencies
    - IQR outlier detection
    - Unknown agent references

    Returns:
        clean_df: Cleaned transaction DataFrame.
        log: Dictionary containing cleaning statistics.
    """

    clean_df = txns.copy()
    agents_df = agents.copy()

    original_rows = len(clean_df)

    # 1. Audit duplicates before removing them
    exact_duplicates = int(clean_df.duplicated().sum())
    duplicate_ids = int(clean_df["txn_id"].duplicated().sum())

    # Keep the first occurrence of each transaction ID
    clean_df = clean_df.drop_duplicates(
        subset=["txn_id"],
        keep="first"
    ).copy()

    # 2. Standardize transaction types
    clean_df["txn_type"] = (
        clean_df["txn_type"]
        .astype("string")
        .str.strip()
        .str.title()
    )

    # 3. Convert amounts to numeric
    clean_df["amount"] = pd.to_numeric(
        clean_df["amount"],
        errors="coerce"
    )

    # Flag missing or invalid amounts
    clean_df["amount_missing"] = clean_df["amount"].isna()

    # Missing financial values are retained as NaN.
    # No artificial transaction amounts are created.

    # 4. Standardize transaction dates
    clean_df["txn_date"] = (
        clean_df["txn_date"]
        .astype("string")
        .str.replace("/", "-", regex=False)
    )

    clean_df["txn_date"] = pd.to_datetime(
        clean_df["txn_date"],
        errors="coerce"
    )

    # 5. Detect outliers using the IQR method
    valid_amounts = clean_df["amount"].dropna()

    if len(valid_amounts) > 0:
        q1 = valid_amounts.quantile(0.25)
        q3 = valid_amounts.quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        clean_df["amount_outlier"] = (
            (clean_df["amount"] < lower_bound) |
            (clean_df["amount"] > upper_bound)
        )
    else:
        clean_df["amount_outlier"] = False

    # 6. Merge agent details, retaining unknown agents
    clean_df = clean_df.merge(
        agents_df,
        on="agent_id",
        how="left",
        validate="many_to_one",
        indicator=True
    )

    unknown_agents = int(
        (clean_df["_merge"] == "left_only").sum()
    )

    clean_df["county"] = clean_df["county"].fillna("Unknown")

    clean_df = clean_df.drop(columns=["_merge"])

    # 7. Create cleaning log
    log = {
        "original_rows": original_rows,
        "exact_duplicate_rows": exact_duplicates,
        "duplicate_ids_removed": duplicate_ids,
        "final_rows": len(clean_df),
        "missing_amounts": int(clean_df["amount_missing"].sum()),
        "outliers_flagged": int(clean_df["amount_outlier"].sum()),
        "unknown_agent_transactions": unknown_agents,
        "invalid_dates": int(clean_df["txn_date"].isna().sum())
    }

    return clean_df, log


Overwriting cleaning_pipeline.py


In [29]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(44)
n = 600

agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(
        ["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20
    ),
    "onboarded": pd.date_range(
        "2024-01-01", periods=20, freq="MS"
    ).strftime("%Y-%m-%d"),
})

txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
    "agent_id": rng.choice(agents["agent_id"], n),
    "txn_type": rng.choice(
        ["Deposit", "withdrawal", "DEPOSIT ", "Withdrawal", "deposit"], n
    ),
    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
    "txn_date": (
        pd.to_datetime("2026-03-01")
        + pd.to_timedelta(rng.integers(0, 60, n), unit="D")
    ).strftime("%Y-%m-%d"),
})

idx = rng.choice(n, 30, replace=False)
txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"
txns.loc[idx[25:], "txn_date"] = (
    txns.loc[idx[25:], "txn_date"].str.replace("-", "/")
)
txns = pd.concat(
    [txns, txns.sample(12, random_state=1)], ignore_index=True
)
txns.loc[rng.choice(len(txns), 8, replace=False), "agent_id"] = "AG099"

agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)

print("Agent datasets generated successfully.")
print("Agents:", len(agents))
print("Transactions:", len(txns))

Agent datasets generated successfully.
Agents: 20
Transactions: 612


In [30]:
import pandas as pd
import numpy as np
import os

from cleaning_pipeline import clean_agent_data

# Load original datasets
agents = pd.read_csv("agents.csv")
txns = pd.read_csv("agent_transactions.csv")

print("Agents dataset shape:", agents.shape)
print("Transactions dataset shape:", txns.shape)

display(agents.head())
display(txns.head())

Agents dataset shape: (20, 3)
Transactions dataset shape: (612, 5)


,agent_id,county,onboarded
0,AG001,Mombasa,2024-01-01
1,AG002,Nairobi,2024-02-01
2,AG003,Kisumu,2024-03-01
3,AG004,Kiambu,2024-04-01
4,AG005,Mombasa,2024-05-01


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17


In [31]:
import os

print("Current directory:", os.getcwd())
print("\nFiles available:")
print(os.listdir())

Current directory: /content

Files available:
['.config', 'agent_transactions.csv', '__pycache__', 'cleaning_pipeline.py', 'outputs', 'agents.csv', 'sample_data']


In [32]:
import os

for filename in [
    "cleaning_pipeline.py",
    "agents.csv",
    "agent_transactions.csv"
]:
    print(filename, os.path.exists(filename))

cleaning_pipeline.py True
agents.csv True
agent_transactions.csv True


In [33]:
# Calculate data quality issues before cleaning

duplicate_count = int(txns.duplicated().sum())

missing_amount_count = int(
    pd.to_numeric(txns["amount"], errors="coerce").isna().sum()
)

inconsistent_type_count = int(
    (~txns["txn_type"].str.strip().str.title().isin(
        ["Deposit", "Withdrawal"]
    )).sum()
)

slash_date_count = int(
    txns["txn_date"].astype(str).str.contains("/", regex=False).sum()
)

unknown_agent_count = int(
    (~txns["agent_id"].isin(agents["agent_id"])).sum()
)

outlier_candidate_count = int(
    (pd.to_numeric(txns["amount"], errors="coerce") == 250000).sum()
)

# Build the audit table
audit = pd.DataFrame([
    {
        "issue": "Exact duplicate transaction records",
        "dimension": "Uniqueness",
        "column": "All columns",
        "rows affected": int(txns.duplicated().sum())
    },
    {
        "issue": "Repeated transaction IDs",
        "dimension": "Uniqueness",
        "column": "txn_id",
        "rows affected": int(txns["txn_id"].duplicated().sum())
    },
    {
        "issue": "Missing or invalid transaction amounts",
        "dimension": "Completeness and validity",
        "column": "amount",
        "rows affected": int(
            pd.to_numeric(txns["amount"], errors="coerce").isna().sum()
        )
    },
    {
        "issue": "Inconsistent transaction type formatting",
        "dimension": "Consistency",
        "column": "txn_type",
        "rows affected": int(
            txns["txn_type"].astype("string").ne(
                txns["txn_type"].astype("string").str.strip().str.title()
            ).sum()
        )
    },
    {
        "issue": "Transaction dates using slash separators",
        "dimension": "Consistency",
        "column": "txn_date",
        "rows affected": int(
            txns["txn_date"].astype(str).str.contains("/", regex=False).sum()
        )
    },
    {
        "issue": "Transactions linked to unknown agents",
        "dimension": "Referential integrity",
        "column": "agent_id",
        "rows affected": int(
            (~txns["agent_id"].isin(agents["agent_id"])).sum()
        )
    },
    {
        "issue": "Potential high-value outliers (KES 250,000)",
        "dimension": "Accuracy",
        "column": "amount",
        "rows affected": int(
            (pd.to_numeric(txns["amount"], errors="coerce") == 250000).sum()
        )
    }
])

display(audit)

,issue,dimension,column,rows affected
0,Exact duplicate transaction records,Uniqueness,All columns,11
1,Repeated transaction IDs,Uniqueness,txn_id,12
2,Missing or invalid transaction amounts,Completeness and validity,amount,22
3,Inconsistent transaction type formatting,Consistency,txn_type,384
4,Transaction dates using slash separators,Consistency,txn_date,5
5,Transactions linked to unknown agents,Referential integrity,agent_id,8
6,"Potential high-value outliers (KES 250,000)",Accuracy,amount,5


### Initial Data Quality Findings

The initial audit identified several data quality issues in the agent transaction dataset.

1. **Uniqueness:** 11 exact duplicate rows and 12 repeated transaction IDs were identified.
2. **Completeness and validity:** 22 transaction amounts were missing or invalid.
3. **Consistency:** 384 transaction types had inconsistent capitalization or whitespace, while 5 transaction dates used slash separators.
4. **Referential integrity:** 8 transactions referenced unknown agent IDs.
5. **Accuracy:** 5 transactions contained amounts of KES 250,000, which were identified as potential outliers requiring investigation.

The dataset requires deduplication, standardization, validation and further investigation before it can be used reliably for financial reporting.

In [34]:
# 1. Investigate transaction type formatting
normalized_types = (
    txns["txn_type"]
    .astype("string")
    .str.strip()
    .str.title()
)

inconsistent_types = int(
    txns["txn_type"].astype("string").ne(normalized_types).sum()
)

print("Original transaction type values:")
print(txns["txn_type"].value_counts(dropna=False))

print("\nRows with inconsistent formatting:", inconsistent_types)


# 2. Investigate duplicate transactions
print("\nDuplicate checks:")
print("Exact duplicate rows:", txns.duplicated().sum())
print("Rows involved in duplicates:", txns.duplicated(keep=False).sum())
print("Duplicate transaction IDs:", txns["txn_id"].duplicated().sum())

Original transaction type values:
txn_type
deposit       142
withdrawal    131
Deposit       128
DEPOSIT       111
Withdrawal    100
Name: count, dtype: int64

Rows with inconsistent formatting: 384

Duplicate checks:
Exact duplicate rows: 11
Rows involved in duplicates: 22
Duplicate transaction IDs: 12


In [35]:
repeated_ids = txns[
    txns["txn_id"].duplicated(keep=False)
].sort_values("txn_id")

display(repeated_ids)

,txn_id,agent_id,txn_type,amount,txn_date
23,TX00024,AG017,DEPOSIT,2798.0,2026-04-28
605,TX00024,AG017,DEPOSIT,2798.0,2026-04-28
66,TX00067,AG002,deposit,NaN,2026-04-07
608,TX00067,AG002,deposit,NaN,2026-04-07
165,TX00166,AG010,deposit,4146.0,2026-04-18
609,TX00166,AG010,deposit,4146.0,2026-04-18
201,TX00202,AG010,Deposit,2243.0,2026-03-22
604,TX00202,AG010,Deposit,2243.0,2026-03-22
611,TX00330,AG016,withdrawal,1219.0,2026-04-07
329,TX00330,AG016,withdrawal,1219.0,2026-04-07


In [36]:
from importlib import reload
import cleaning_pipeline

reload(cleaning_pipeline)

from cleaning_pipeline import clean_agent_data

# Clean the original datasets
cleaned_df, cleaning_log = clean_agent_data(txns, agents)

print("CLEANING LOG")
for key, value in cleaning_log.items():
    print(f"{key}: {value}")

print("\nCleaned dataset shape:", cleaned_df.shape)

display(cleaned_df.head(10))

CLEANING LOG
original_rows: 612
exact_duplicate_rows: 11
duplicate_ids_removed: 12
final_rows: 600
missing_amounts: 20
outliers_flagged: 40
unknown_agent_transactions: 7
invalid_dates: 0

Cleaned dataset shape: (600, 9)


,txn_id,agent_id,txn_type,amount,txn_date,amount_missing,amount_outlier,county,onboarded
0,TX00001,AG009,Deposit,353.0,2026-03-03,False,False,Nakuru,2024-09-01
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False,False,Mombasa,2025-08-01
2,TX00003,AG008,Deposit,9129.0,2026-04-29,False,True,Kisumu,2024-08-01
3,TX00004,AG009,Deposit,652.0,2026-04-24,False,False,Nakuru,2024-09-01
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False,False,Nakuru,2024-09-01
5,TX00006,AG019,Withdrawal,931.0,2026-04-23,False,False,Mombasa,2025-07-01
6,TX00007,AG020,Deposit,2583.0,2026-04-02,False,False,Mombasa,2025-08-01
7,TX00008,AG019,Deposit,1797.0,2026-04-12,False,False,Mombasa,2025-07-01
8,TX00009,AG012,Withdrawal,4184.0,2026-04-04,False,False,Kisumu,2024-12-01
9,TX00010,AG015,Deposit,1987.0,2026-04-17,False,False,Nairobi,2025-03-01


In [37]:
# Update the deduplication logic in cleaning_pipeline.py

with open("cleaning_pipeline.py", "r") as file:
    code = file.read()

old_code = '''    duplicates_removed = int(clean_df.duplicated().sum())
    clean_df = clean_df.drop_duplicates().copy()
'''

new_code = '''    exact_duplicates = int(clean_df.duplicated().sum())
    duplicate_ids_removed = int(clean_df["txn_id"].duplicated().sum())

    clean_df = clean_df.drop_duplicates(
        subset=["txn_id"],
        keep="first"
    ).copy()

    duplicates_removed = duplicate_ids_removed
'''

if old_code not in code:
    print("Original code section not found. No changes made.")
else:
    code = code.replace(old_code, new_code)

    code = code.replace(
        '"duplicates_removed": duplicates_removed,',
        '"exact_duplicate_rows": exact_duplicates,\\n'
        '        "duplicate_ids_removed": duplicates_removed,'
    )

    with open("cleaning_pipeline.py", "w") as file:
        file.write(code)

    print("Cleaning pipeline updated successfully.")

Original code section not found. No changes made.


In [38]:
import importlib
import cleaning_pipeline

importlib.reload(cleaning_pipeline)

from cleaning_pipeline import clean_agent_data

cleaned_df, cleaning_log = clean_agent_data(txns, agents)

print("CLEANING LOG")

for key, value in cleaning_log.items():
    print(f"{key}: {value}")

print("\nFinal dataset shape:", cleaned_df.shape)

print(
    "Repeated transaction IDs remaining:",
    cleaned_df["txn_id"].duplicated().sum()
)

CLEANING LOG
original_rows: 612
exact_duplicate_rows: 11
duplicate_ids_removed: 12
final_rows: 600
missing_amounts: 20
outliers_flagged: 40
unknown_agent_transactions: 7
invalid_dates: 0

Final dataset shape: (600, 9)
Repeated transaction IDs remaining: 0


In [39]:
high_value_transactions = cleaned_df.loc[
    cleaned_df["amount"] == 250000,
    ["txn_id", "agent_id", "amount", "txn_date", "county"]
]

print("Number of KES 250,000 transactions:",
      len(high_value_transactions))

display(high_value_transactions)

Number of KES 250,000 transactions: 5


,txn_id,agent_id,amount,txn_date,county
306,TX00307,AG012,250000.0,2026-03-02,Kisumu
315,TX00316,AG014,250000.0,2026-04-01,Nairobi
424,TX00425,AG005,250000.0,2026-03-02,Mombasa
463,TX00464,AG009,250000.0,2026-04-11,Nakuru
594,TX00595,AG016,250000.0,2026-03-11,Kiambu


### Investigation of High-Value Transactions

The investigation identified **5 transactions valued at KES 250,000 each**. These transactions occurred across Kisumu, Nairobi, Mombasa, Nakuru and Kiambu counties.

The transactions were retained in the cleaned dataset because their high values alone do not prove that they are incorrect.

The IQR method flagged 40 potential outliers in total. These five KES 250,000 transactions are a specific group of high-value records and should be verified against the original transaction records.

**Recommendation:** The relevant agents or transaction records should be reviewed to confirm the legitimacy of these amounts before any adjustments are made.

In [40]:
unknown_agent_transactions = cleaned_df[
    cleaned_df["county"] == "Unknown"
]

print("Number of transactions linked to unknown agents:",
      len(unknown_agent_transactions))

display(
    unknown_agent_transactions[
        ["txn_id", "agent_id", "txn_type", "amount", "txn_date", "county"]
    ]
)

Number of transactions linked to unknown agents: 7


,txn_id,agent_id,txn_type,amount,txn_date,county
55,TX00056,AG099,Deposit,8507.0,2026-04-01,Unknown
68,TX00069,AG099,Withdrawal,1877.0,2026-04-24,Unknown
77,TX00078,AG099,Deposit,1770.0,2026-04-06,Unknown
237,TX00238,AG099,Deposit,4944.0,2026-04-06,Unknown
267,TX00268,AG099,Deposit,3110.0,2026-04-09,Unknown
444,TX00445,AG099,Deposit,5309.0,2026-04-27,Unknown
557,TX00558,AG099,Deposit,1639.0,2026-03-22,Unknown


### Investigation of Unknown Agent References

The cleaned dataset contains **7 transactions linked to agent ID AG099**, which does not exist in the supplied agent master dataset.

These transactions were retained to preserve financial records. Their county values were assigned `Unknown` because the agent's county could not be established from the available master data.

**Recommendation:** The operations team should verify whether AG099 is a legitimate agent and update the agent master dataset with the correct details. The transactions should not be assigned to an arbitrary county or deleted.

In [41]:
county_summary = (
    cleaned_df.groupby(["county", "txn_type"], dropna=False)
    .agg(
        total_transaction_value=("amount", "sum"),
        transaction_count=("txn_id", "count"),
        average_transaction_value=("amount", "mean")
    )
    .reset_index()
)

display(county_summary)

,county,txn_type,total_transaction_value,transaction_count,average_transaction_value
0,Kiambu,Deposit,334737.0,43,7784.581395
1,Kiambu,Withdrawal,75274.0,27,2895.153846
2,Kisumu,Deposit,431091.0,74,6071.704225
3,Kisumu,Withdrawal,74598.0,37,2072.166667
4,Mombasa,Deposit,141880.0,72,2086.470588
5,Mombasa,Withdrawal,368592.0,61,6042.491803
6,Nairobi,Deposit,536484.0,118,4706.000000
7,Nairobi,Withdrawal,154930.0,62,2671.206897
8,Nakuru,Deposit,181337.0,60,3126.500000
9,Nakuru,Withdrawal,326818.0,39,8600.473684


In [42]:
cleaned_df["week"] = (
    cleaned_df["txn_date"]
    .dt.to_period("W")
    .astype(str)
)

weekly_pivot = pd.pivot_table(
    cleaned_df,
    index="county",
    columns="week",
    values="amount",
    aggfunc="sum",
    fill_value=0
)

display(weekly_pivot)

week,2026-02-23/2026-03-01,2026-03-02/2026-03-08,2026-03-09/2026-03-15,2026-03-16/2026-03-22,2026-03-23/2026-03-29,2026-03-30/2026-04-05,2026-04-06/2026-04-12,2026-04-13/2026-04-19,2026-04-20/2026-04-26,2026-04-27/2026-05-03
county,,,,,,,,,,
Kiambu,0.0,18317.0,265995.0,22054.0,11504.0,22609.0,15030.0,12265.0,24507.0,17730.0
Kisumu,1893.0,287933.0,23186.0,24572.0,27077.0,41077.0,15341.0,25863.0,37439.0,21308.0
Mombasa,6759.0,283893.0,28701.0,23368.0,17050.0,38344.0,45316.0,26730.0,16687.0,23624.0
Nairobi,7046.0,68967.0,51084.0,41331.0,39597.0,282931.0,55010.0,54656.0,61634.0,29158.0
Nakuru,967.0,56574.0,28730.0,39795.0,17222.0,18145.0,276609.0,24376.0,30862.0,14875.0
Unknown,0.0,0.0,0.0,1639.0,0.0,8507.0,9824.0,0.0,1877.0,5309.0


In [43]:
import os

os.makedirs("outputs", exist_ok=True)

cleaned_df.to_csv(
    "outputs/agent_transactions_clean.csv",
    index=False
)

print("Cleaned dataset exported successfully!")

print(
    "File exists:",
    os.path.exists("outputs/agent_transactions_clean.csv")
)

print("Exported rows:", len(cleaned_df))

Cleaned dataset exported successfully!
File exists: True
Exported rows: 600


In [44]:
print("POST-CLEANING DATA QUALITY CHECK")
print("-" * 45)

print("Total transactions:", len(cleaned_df))
print("Duplicate transaction IDs:",
      cleaned_df["txn_id"].duplicated().sum())

print("Missing transaction amounts:",
      cleaned_df["amount"].isna().sum())

print("Invalid transaction dates:",
      cleaned_df["txn_date"].isna().sum())

print("Unknown agent transactions:",
      (cleaned_df["county"] == "Unknown").sum())

print("\nTransaction types:")
print(cleaned_df["txn_type"].value_counts())

print("\nCounties:")
print(cleaned_df["county"].value_counts())

POST-CLEANING DATA QUALITY CHECK
---------------------------------------------
Total transactions: 600
Duplicate transaction IDs: 0
Missing transaction amounts: 20
Invalid transaction dates: 0
Unknown agent transactions: 7

Transaction types:
txn_type
Deposit       373
Withdrawal    227
Name: count, dtype: Int64

Counties:
county
Nairobi    180
Mombasa    133
Kisumu     111
Nakuru      99
Kiambu      70
Unknown      7
Name: count, dtype: int64


# Data Quality Report – Agent Transaction Dataset

## 1. Project Overview

The objective of this exercise was to audit and clean an agent transaction dataset containing 612 records from 20 registered agents. The cleaning process addressed duplicate transactions, inconsistent transaction types, missing financial values, date inconsistencies, potential outliers and unknown agent references.

## 2. Data Quality Findings and Actions

| Data quality dimension | Initial findings | Cleaning action and outcome |
|---|---|---|
| Uniqueness | 11 exact duplicate rows and 12 repeated transaction IDs | Removed repeated transaction IDs, retaining the first occurrence. Final dataset contains 600 unique transactions. |
| Completeness | Missing or invalid transaction amounts | Converted invalid amounts to missing values. Retained 20 records with missing amounts for investigation. |
| Consistency | Inconsistent transaction type formatting and 5 dates using slash separators | Standardized transaction types and date formats. |
| Validity | Invalid or unparseable transaction dates | No invalid dates remained after standardization. |
| Accuracy | 40 potential outliers identified using the IQR method, including 5 transactions valued at KES 250,000 | Retained and flagged potential outliers for verification instead of automatically deleting or modifying financial records. |
| Referential integrity | 8 initial transactions referenced unknown agent IDs | After deduplication, 7 transactions linked to AG099 remained. Their county was assigned Unknown. |

## 3. Final Dataset Validation

- Original transaction records: 612
- Final transaction records: 600
- Duplicate transaction IDs remaining: 0
- Missing transaction amounts: 20
- Invalid transaction dates: 0
- Unknown agent references: 7
- Deposit transactions: 373
- Withdrawal transactions: 227

## 4. Key Business Observations

- Nairobi recorded the highest deposit value at KES 536,484.
- Kisumu recorded the second-highest deposit value at KES 431,091.
- Mombasa recorded the highest withdrawal value at KES 368,592.
- Nakuru recorded the highest average withdrawal value at approximately KES 8,600.47.
- Weekly transaction summaries revealed periods of unusually high activity across counties, which may require further investigation.

## 5. Recommendations

1. Investigate the 20 transactions with missing amounts using original transaction records.
2. Verify agent ID AG099 and update the agent master dataset if the agent is legitimate.
3. Review the five KES 250,000 transactions to establish whether they are legitimate high-value transactions.
4. Introduce automated validation rules for transaction amounts, dates and agent IDs.
5. Perform regular duplicate checks before accepting new transaction records.

## 6. Conclusion

The cleaning pipeline successfully transformed the original 612-row dataset into a 600-row cleaned dataset with unique transaction IDs, standardized transaction types and valid transaction dates.

Missing financial amounts, potential outliers and unknown agent references were retained and flagged for further investigation to preserve financial data integrity.

**Data privacy:** The dataset contains agent identifiers and financial transaction information. It should be handled securely, shared only with authorized personnel and stored in access-controlled locations. Personal or sensitive transaction details should not be unnecessarily exposed in public reports or repositories.

In [45]:
import os

for root, dirs, files in os.walk("/content"):
    dirs[:] = [d for d in dirs if d not in [".config", "sample_data", ".ipynb_checkpoints"]]

    for file in files:
        print(os.path.relpath(os.path.join(root, file), "/content"))

agent_transactions.csv
cleaning_pipeline.py
agents.csv
__pycache__/cleaning_pipeline.cpython-313.pyc
outputs/agent_transactions_clean.csv
